# 🔬 Chronos-2 Evaluation for RUL Prediction — FD002 (Multi-Condition)

**Objective**: Extend the Chronos-2 RUL evaluation to **C-MAPSS FD002**, which contains **6 operating conditions**.

**Key difference vs FD001**: raw sensor readings depend on the operating regime, not only on degradation. We therefore apply **per-regime (condition-specific) normalization**: the 3 operating settings are clustered into 6 regimes, and each sensor is normalized *within* its regime so that only the degradation signal remains.

**Structure**:
- **Part A**: Data prep — load FD002, identify 6 regimes, per-regime normalization, feature selection, windows
- **Part B**: Chronos-1 embeddings (baseline)
- **Part C**: Chronos-2 embeddings + pooling strategies + regression heads
- **Part D**: Results & visualization

**Author**: Fatima Khadija Benzine

---
## Setup

In [ ]:
# Install dependencies
!pip install -q 'chronos-forecasting>=2.2' xgboost torch

import os
!rm -rf /content/RUL-Chronos
!git clone https://github.com/f-khadija-benzine/RUL-Chronos.git /content/RUL-Chronos

os.chdir('/content/RUL-Chronos/src')

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import json, time, warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

from data_loader import MultiDatasetLoader
from preprocessing import DataNormalizer

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = f'/content/drive/MyDrive/PhD_results/Chronos_Eval_{TIMESTAMP}'
os.makedirs(SAVE_DIR, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"Save directory: {SAVE_DIR}")
print("Setup complete ✓")

---
## Part A — Data Preparation (FD002)

In [ ]:
# === A1: Load C-MAPSS FD002 ===
DATASET = 'FD002'          # 6 operating conditions
W = 30
RUL_CAP = 125
N_REGIMES = 6              # FD002 has 6 operating regimes

loader = MultiDatasetLoader()
ds = loader.load_cmapss_dataset(DATASET)

train_raw = ds['train'].copy()
test_raw = ds['test'].copy()

# Apply RUL cap
train_raw['rul'] = train_raw['rul'].clip(upper=RUL_CAP)
if 'rul' in test_raw.columns:
    test_raw['rul'] = test_raw['rul'].clip(upper=RUL_CAP)

sensor_cols = [c for c in train_raw.columns if c.startswith('sensor_')]
setting_cols = [c for c in train_raw.columns if c.startswith('setting_')]

print(f"✓ Data loaded: {DATASET}")
print(f"  Train: {len(train_raw)} samples, {train_raw['unit'].nunique()} units")
print(f"  Test:  {len(test_raw)} samples, {test_raw['unit'].nunique()} units")
print(f"  Sensors: {len(sensor_cols)}, Settings: {len(setting_cols)}")

In [ ]:
# === A2a: Identify the 6 Operating Regimes ===
# Cluster the 3 operating settings into 6 groups. Fit on TRAIN only,
# then assign the same clusters to TEST (no leakage).

kmeans = KMeans(n_clusters=N_REGIMES, random_state=42, n_init=10)
train_raw['regime'] = kmeans.fit_predict(train_raw[setting_cols].values)
test_raw['regime']  = kmeans.predict(test_raw[setting_cols].values)

print(f"✓ Identified {N_REGIMES} operating regimes")
print("\nRegime distribution (train):")
print(train_raw['regime'].value_counts().sort_index().to_string())
print("\nRegime distribution (test):")
print(test_raw['regime'].value_counts().sort_index().to_string())

In [ ]:
# === A2b: Visualize the 6 Operating Regimes ===
# Confirm the operating settings separate cleanly into 6 clusters.

fig = plt.figure(figsize=(14, 4))
colors = plt.cm.tab10(np.linspace(0, 1, N_REGIMES))

# 3D-style view via three 2D projections of the settings
pairs = [(0, 1), (0, 2), (1, 2)]
for idx, (a, b) in enumerate(pairs):
    ax = fig.add_subplot(1, 3, idx + 1)
    for r in range(N_REGIMES):
        mask = train_raw['regime'] == r
        ax.scatter(train_raw.loc[mask, setting_cols[a]],
                   train_raw.loc[mask, setting_cols[b]],
                   s=6, alpha=0.4, color=colors[r], label=f'Regime {r}')
    ax.set_xlabel(setting_cols[a])
    ax.set_ylabel(setting_cols[b])
    ax.set_title(f'{setting_cols[a]} vs {setting_cols[b]}')
    if idx == 2:
        ax.legend(markerscale=2, fontsize=8, loc='center left',
                  bbox_to_anchor=(1.02, 0.5))

plt.suptitle('FD002 Operating Regimes (K-Means, k=6)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fd002_regimes_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ The 6 regimes should appear as clearly separated clusters.")

In [ ]:
# === A2c: Per-Regime (Condition-Specific) Normalization ===
# Fit one MinMax scaler PER regime on sensors, using TRAIN only,
# then apply to both train and test. This removes the regime effect
# so the normalized signal reflects degradation, not operating condition.

scalers = {}
train_norm = train_raw.copy()
test_norm  = test_raw.copy()

for r in range(N_REGIMES):
    scaler = MinMaxScaler()
    tr_mask = train_raw['regime'] == r
    scaler.fit(train_raw.loc[tr_mask, sensor_cols].values)
    scalers[r] = scaler

    train_norm.loc[tr_mask, sensor_cols] = scaler.transform(
        train_raw.loc[tr_mask, sensor_cols].values)

    te_mask = test_raw['regime'] == r
    if te_mask.any():
        test_norm.loc[te_mask, sensor_cols] = scaler.transform(
            test_raw.loc[te_mask, sensor_cols].values)

# After per-regime normalization, sensors are the feature pool.
# (Settings only encoded the regime, which is now normalized out.)
feature_cols = sensor_cols

print("✓ Per-regime MinMax normalization complete")
print(f"  Feature pool: {len(feature_cols)} sensors")

# Sanity check: a sensor should now look like a degradation trend,
# not a step function jumping between regimes.
u = sorted(train_norm['unit'].unique())[0]
ud = train_norm[train_norm['unit'] == u].sort_values('cycle')
plt.figure(figsize=(12, 3))
for s in sensor_cols[:4]:
    plt.plot(ud['cycle'], ud[s], label=s, alpha=0.8)
plt.xlabel('Cycle'); plt.ylabel('Normalized value')
plt.title(f'Normalized sensors for unit {u} (should show smooth degradation trends)')
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fd002_normcheck_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === A3: Feature Selection ===

def variance_filter(df, feature_cols, threshold=0.001):
    """Remove low-variance features"""
    variances = df[feature_cols].var()
    selected = variances[variances > threshold].index.tolist()
    removed = [f for f in feature_cols if f not in selected]
    return selected, removed

def correlation_filter(df, feature_cols, threshold=0.95):
    """Remove highly correlated features"""
    corr_matrix = df[feature_cols].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    selected = [f for f in feature_cols if f not in to_drop]
    return selected, to_drop

def aficv_selection(df, feature_cols, target_col='rul', threshold=0.90):
    """AFICv: Accumulated Feature Importance with Cross-Validation"""
    X = df[feature_cols].values
    y = df[target_col].values

    model = XGBRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    model.fit(X, y)

    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]

    cumsum = np.cumsum(importances[indices])
    n_features = np.searchsorted(cumsum, threshold) + 1

    selected_indices = indices[:n_features]
    selected = [feature_cols[i] for i in selected_indices]

    return selected, cumsum[n_features-1]

# Strategy 1: Variance + Correlation filtering
print("=== Strategy 1: Variance + Correlation Filtering ===")
feats_var, removed_var = variance_filter(train_norm, feature_cols, threshold=0.001)
print(f"  Variance filter: {len(feature_cols)} → {len(feats_var)} (removed: {removed_var})")

feats_corr, removed_corr = correlation_filter(train_norm, feats_var, threshold=0.95)
print(f"  Correlation filter: {len(feats_var)} → {len(feats_corr)} (removed: {removed_corr})")

# Strategy 2: AFICv
print("\n=== Strategy 2: AFICv (90% importance coverage) ===")
feats_aficv, coverage = aficv_selection(train_norm, feature_cols, threshold=0.90)
print(f"  Selected: {len(feats_aficv)} features (coverage: {coverage:.1%})")
print(f"  Features: {feats_aficv}")

# Store feature sets
feature_sets = {
    'correlation': feats_corr,
    'aficv': feats_aficv
}

print("\n=== Summary ===")
for name, feats in feature_sets.items():
    print(f"  {name}: {len(feats)} features")

In [ ]:
# === A4: Create Sliding Windows ===

def create_sliding_windows(df, feature_cols, window_size=30):
    """Create sliding windows for each unit"""
    X_list, y_list, unit_list = [], [], []

    for unit in sorted(df['unit'].unique()):
        unit_data = df[df['unit'] == unit].sort_values('cycle')
        n_samples = len(unit_data)

        if n_samples < window_size:
            continue

        values = unit_data[feature_cols].values
        ruls = unit_data['rul'].values

        for start in range(n_samples - window_size + 1):
            end = start + window_size
            X_list.append(values[start:end])
            y_list.append(ruls[end - 1])
            unit_list.append(unit)

    return np.array(X_list), np.array(y_list), np.array(unit_list)

# Create windows for each feature set
data_dict = {}

for fs_name, features in feature_sets.items():
    X_train, y_train, units_train = create_sliding_windows(train_norm, features, W)
    X_test, y_test, units_test = create_sliding_windows(test_norm, features, W)

    data_dict[fs_name] = {
        'X_train': X_train, 'y_train': y_train, 'units_train': units_train,
        'X_test': X_test, 'y_test': y_test, 'units_test': units_test,
        'features': features
    }

    print(f"{fs_name}: X_train={X_train.shape}, X_test={X_test.shape}")

print("\n✓ Sliding windows created")

In [ ]:
# === A5: Evaluation Helpers ===

def nasa_score(y_true, y_pred):
    """NASA asymmetric scoring function (Saxena et al. 2008)"""
    d = y_pred - y_true
    score = np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1)
    return np.sum(score)

def evaluate_per_unit(y_true, y_pred, unit_labels):
    """Evaluate using last prediction per unit (standard C-MAPSS protocol)"""
    preds_last, true_last = [], []
    for u in sorted(set(unit_labels)):
        mask = unit_labels == u
        if mask.sum() > 0:
            preds_last.append(y_pred[mask][-1])
            true_last.append(y_true[mask][-1])

    preds_last = np.array(preds_last)
    true_last = np.array(true_last)

    rmse = np.sqrt(mean_squared_error(true_last, preds_last))
    score = nasa_score(true_last, preds_last)
    return rmse, score

ALL_RESULTS = []
print("Evaluation helpers ✓")

In [ ]:
# === A6: Device ===
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Part B — Chronos-1 (Univariate Baseline)

In [10]:
# === B1: Load Chronos-1 Pipeline ===
from chronos import ChronosPipeline

CHRONOS1_MODEL = "amazon/chronos-t5-small"

print(f"Loading Chronos-1: {CHRONOS1_MODEL}...")
chronos1_pipeline = ChronosPipeline.from_pretrained(
    CHRONOS1_MODEL,
    device_map=DEVICE,
    torch_dtype=torch.float32
)
print("Chronos-1 loaded ✓")
print(f"  Architecture: T5 encoder-decoder")
print(f"  Capability: Univariate only")

Loading Chronos-1: amazon/chronos-t5-small...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Chronos-1 loaded ✓
  Architecture: T5 encoder-decoder
  Capability: Univariate only


In [11]:
# === B2: Extract Chronos-1 Embeddings ===

def extract_chronos1_embeddings(pipeline, X, batch_size=64):
    all_embeddings = []
    device = pipeline.model.device   # device réel du modèle

    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]  # (B, W, features)

        batch_embeddings = []
        for feat_idx in range(batch.shape[2]):
            series = batch[:, :, feat_idx]  # (B, W)
            ts_tensor = torch.tensor(series, dtype=torch.float32).to(device)  # ← .to(device)

            with torch.no_grad():
                emb = pipeline.embed(ts_tensor)[0]      # [0] pour le tuple
                emb_pooled = emb.mean(dim=1).cpu().numpy()

            batch_embeddings.append(emb_pooled)

        # concat les features
        batch_emb = np.concatenate(batch_embeddings, axis=1)
        all_embeddings.append(batch_emb)

    return np.vstack(all_embeddings)

print("Chronos-1 embedding function defined ✓")

Chronos-1 embedding function defined ✓


In [12]:
import inspect
print('.to(device)' in inspect.getsource(extract_chronos1_embeddings))

True


In [13]:
# === B3: Chronos-1 Evaluation ===

chronos1_results = {}

for fs_name in feature_sets.keys():
    print(f"\n{'='*60}")
    print(f"Chronos-1 + {fs_name.upper()} ({len(feature_sets[fs_name])} features)")
    print('='*60)

    data = data_dict[fs_name]

    print("Extracting train embeddings...")
    start_time = time.time()
    X_train_emb = extract_chronos1_embeddings(chronos1_pipeline, data['X_train'])
    train_time = time.time() - start_time
    print(f"  Shape: {X_train_emb.shape}, Time: {train_time:.1f}s")

    print("Extracting test embeddings...")
    start_time = time.time()
    X_test_emb = extract_chronos1_embeddings(chronos1_pipeline, data['X_test'])
    test_time = time.time() - start_time
    print(f"  Shape: {X_test_emb.shape}, Time: {test_time:.1f}s")

    heads = [
        ('Ridge', Ridge(alpha=1.0)),
        ('MLP', MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=500, random_state=42)),
        ('XGBoost', XGBRegressor(n_estimators=100, max_depth=5, random_state=42))
    ]

    for head_name, head_model in heads:
        print(f"\n  Training {head_name} head...")
        head_model.fit(X_train_emb, data['y_train'])

        y_pred = head_model.predict(X_test_emb)
        rmse, score = evaluate_per_unit(data['y_test'], y_pred, data['units_test'])

        result = {
            'Model': 'Chronos-1',
            'Feature_Set': fs_name,
            'Head': head_name,
            'Test_RMSE': round(rmse, 2),
            'Test_Score': round(score, 1),
            'Time_sec': round(train_time + test_time, 1)
        }
        ALL_RESULTS.append(result)
        chronos1_results[f"{fs_name}_{head_name}"] = result

        print(f"    RMSE: {rmse:.2f}, Score: {score:.1f}")

print("\n✓ Chronos-1 evaluation complete")


Chronos-1 + CORRELATION (17 features)
Extracting train embeddings...


RuntimeError: Expected all tensors to be on the same device, but got boundaries is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_Tensor_bucketize)

---
## Part C — Chronos-2 (Multivariate)

In [ ]:
# === C1: Load Chronos-2 Pipeline ===
from chronos import Chronos2Pipeline

CHRONOS2_MODEL = "amazon/chronos-2"

print(f"Loading Chronos-2: {CHRONOS2_MODEL}...")
chronos2_pipeline = Chronos2Pipeline.from_pretrained(
    CHRONOS2_MODEL,
    device_map=DEVICE
)
print("Chronos-2 loaded ✓")
print(f"  Parameters: 120M")
print(f"  Architecture: Encoder-only + Group Attention")
print(f"  Capabilities: Univariate, Multivariate")

In [ ]:
# === C2: Extract Chronos-2 Embeddings (Multivariate) ===

def extract_chronos2_embeddings(pipeline, X, batch_size=32):
    """
    Extract Chronos-2 encoder embeddings.
    Chronos-2 can process multivariate series natively.
    X shape: (n_samples, window_size, n_features)
    """
    n_samples, window_size, n_features = X.shape
    all_embeddings = []

    for i in range(0, n_samples, batch_size):
        batch = X[i:i+batch_size]
        batch_t = np.transpose(batch, (0, 2, 1))

        ts_tensor = torch.tensor(batch_t, dtype=torch.float32)

        with torch.no_grad():
            result = pipeline.embed(ts_tensor)
            # embed() returns (embeddings, loc_scale) tuple
            emb = result[0] if isinstance(result, tuple) else result
            emb_pooled = emb.mean(dim=1).cpu().numpy()

        all_embeddings.append(emb_pooled)

        if (i // batch_size) % 10 == 0:
            print(f"    Processed {min(i+batch_size, n_samples)}/{n_samples} samples")

    return np.vstack(all_embeddings)

print("Chronos-2 embedding function defined ✓")

In [ ]:
# === C2b: Different Embedding Pooling Strategies ===

def extract_chronos2_embeddings_v2(pipeline, X, batch_size=32, pooling='mean'):
    """
    Extract Chronos-2 embeddings with different pooling strategies.

    Args:
        pooling: 'mean' | 'last' | 'concat'
            - mean: Average over features and patches (current approach)
            - last: Take only the last patch embedding
            - concat: Concatenate all patch embeddings (larger dim)
    """
    n_samples, window_size, n_features = X.shape
    all_embeddings = []

    for i in range(0, n_samples, batch_size):
        batch = X[i:i+batch_size]
        batch_t = np.transpose(batch, (0, 2, 1))  # (batch, features, window)
        ts_tensor = torch.tensor(batch_t, dtype=torch.float32)

        with torch.no_grad():
            result = pipeline.embed(ts_tensor)
            emb_list = result[0]  # list of tensors
            emb = torch.stack(emb_list)  # (batch, n_features, n_patches, embed_dim)

            if pooling == 'mean':
                # Current: mean over features and patches -> (batch, embed_dim)
                emb_pooled = emb.mean(dim=(1, 2)).cpu().numpy()

            elif pooling == 'last':
                # Take last patch only, mean over features -> (batch, embed_dim)
                emb_pooled = emb[:, :, -1, :].mean(dim=1).cpu().numpy()

            elif pooling == 'concat':
                # Concatenate all patches, mean over features -> (batch, n_patches * embed_dim)
                batch_size_actual = emb.shape[0]
                n_patches = emb.shape[2]
                embed_dim = emb.shape[3]
                # Mean over features first -> (batch, n_patches, embed_dim)
                emb_feat_mean = emb.mean(dim=1)
                # Flatten patches -> (batch, n_patches * embed_dim)
                emb_pooled = emb_feat_mean.reshape(batch_size_actual, -1).cpu().numpy()

            elif pooling == 'last_concat':
                # Concatenate last patch from all features -> (batch, n_features * embed_dim)
                emb_pooled = emb[:, :, -1, :].reshape(emb.shape[0], -1).cpu().numpy()

        all_embeddings.append(emb_pooled)

    return np.vstack(all_embeddings)

print("Pooling strategies defined:")
print("  - mean: (batch, 768) - current approach")
print("  - last: (batch, 768) - last patch only")
print("  - concat: (batch, n_patches * 768) - all patches concatenated")
print("  - last_concat: (batch, n_features * 768) - last patch per feature")

In [ ]:
# === C3: Chronos-2 Evaluation ===

# Initialize prediction storage
all_predictions = {}

chronos2_results = {}

for fs_name in feature_sets.keys():
    print(f"\n{'='*60}")
    print(f"Chronos-2 + {fs_name.upper()} ({len(feature_sets[fs_name])} features)")
    print('='*60)

    # Initialize storage for this feature set
    all_predictions[fs_name] = {}

    data = data_dict[fs_name]

    print("Extracting train embeddings (multivariate)...")
    start_time = time.time()
    X_train_emb = extract_chronos2_embeddings(chronos2_pipeline, data['X_train'])
    train_time = time.time() - start_time
    print(f"  Shape: {X_train_emb.shape}, Time: {train_time:.1f}s")

    print("Extracting test embeddings...")
    start_time = time.time()
    X_test_emb = extract_chronos2_embeddings(chronos2_pipeline, data['X_test'])
    test_time = time.time() - start_time
    print(f"  Shape: {X_test_emb.shape}, Time: {test_time:.1f}s")

    heads = [
        ('Ridge', Ridge(alpha=1.0)),
        ('MLP', MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=500, random_state=456)),
        ('XGBoost', XGBRegressor(n_estimators=100, max_depth=5, random_state=456))
    ]

    for head_name, head_model in heads:
        print(f"\n  Training {head_name} head...")
        head_model.fit(X_train_emb, data['y_train'])

        y_pred = head_model.predict(X_test_emb)

        # Store predictions for visualization
        all_predictions[fs_name][head_name] = {
            'y_pred': y_pred.copy(),
            'y_test': data['y_test'].copy(),
            'units_test': data['units_test'].copy()
        }

        rmse, score = evaluate_per_unit(data['y_test'], y_pred, data['units_test'])

        result = {
            'Model': 'Chronos-2',
            'Feature_Set': fs_name,
            'Head': head_name,
            'Test_RMSE': round(rmse, 2),
            'Test_Score': round(score, 1),
            'Time_sec': round(train_time + test_time, 1)
        }
        ALL_RESULTS.append(result)
        chronos2_results[f"{fs_name}_{head_name}"] = result

        print(f"    RMSE: {rmse:.2f}, Score: {score:.1f}")

print("\n✓ Chronos-2 evaluation complete")
print(f"✓ Predictions stored for visualization: {list(all_predictions.keys())}")

In [ ]:
# === C2c: Evaluate All Pooling Strategies ===

pooling_strategies = ['mean', 'last', 'concat', 'last_concat']
pooling_results = []

fs_name = 'correlation'
data = data_dict[fs_name]

for pooling in pooling_strategies:
    print(f"\n{'='*60}")
    print(f"Pooling: {pooling.upper()}")
    print('='*60)

    # Extract embeddings
    print("  Extracting train embeddings...")
    X_train_emb = extract_chronos2_embeddings_v2(chronos2_pipeline, data['X_train'], pooling=pooling)
    print(f"    Shape: {X_train_emb.shape}")

    print("  Extracting test embeddings...")
    X_test_emb = extract_chronos2_embeddings_v2(chronos2_pipeline, data['X_test'], pooling=pooling)
    print(f"    Shape: {X_test_emb.shape}")

    # Test with MLP (best head so far)
    print("  Training MLP...")
    mlp = MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=500,
                       early_stopping=True, random_state=42)
    mlp.fit(X_train_emb, data['y_train'])
    y_pred = mlp.predict(X_test_emb)
    rmse, score = evaluate_per_unit(data['y_test'], y_pred, data['units_test'])

    print(f"  ✓ RMSE: {rmse:.2f}, Score: {score:.1f}")

    pooling_results.append({
        'Pooling': pooling,
        'Embed_Dim': X_train_emb.shape[1],
        'RMSE': round(rmse, 2),
        'Score': round(score, 1)
    })

# Summary
print(f"\n{'='*60}")
print("POOLING STRATEGY COMPARISON (Chronos-2 + MLP)")
print('='*60)
pooling_df = pd.DataFrame(pooling_results).sort_values('RMSE')
print(pooling_df.to_string(index=False))

In [ ]:
# === C2d: Evaluate All Pooling Strategies with XGBoost ===

pooling_strategies = ['mean', 'last', 'concat', 'last_concat']
pooling_results_xgb = []

fs_name = 'correlation'
data = data_dict[fs_name]

for pooling in pooling_strategies:
    print(f"\n{'='*60}")
    print(f"Pooling: {pooling.upper()} + XGBoost")
    print('='*60)

    # Extract embeddings
    print("  Extracting train embeddings...")
    X_train_emb = extract_chronos2_embeddings_v2(chronos2_pipeline, data['X_train'], pooling=pooling)
    print(f"    Shape: {X_train_emb.shape}")

    print("  Extracting test embeddings...")
    X_test_emb = extract_chronos2_embeddings_v2(chronos2_pipeline, data['X_test'], pooling=pooling)
    print(f"    Shape: {X_test_emb.shape}")

    # Test with XGBoost
    print("  Training XGBoost...")
    xgb = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1,
                       random_state=42, verbosity=0)
    xgb.fit(X_train_emb, data['y_train'])
    y_pred = xgb.predict(X_test_emb)
    rmse, score = evaluate_per_unit(data['y_test'], y_pred, data['units_test'])

    print(f"  ✓ RMSE: {rmse:.2f}, Score: {score:.1f}")

    pooling_results_xgb.append({
        'Pooling': pooling,
        'Head': 'XGBoost',
        'Embed_Dim': X_train_emb.shape[1],
        'RMSE': round(rmse, 2),
        'Score': round(score, 1)
    })

# Summary
print(f"\n{'='*60}")
print("POOLING STRATEGY COMPARISON (Chronos-2 + XGBoost)")
print('='*60)
pooling_xgb_df = pd.DataFrame(pooling_results_xgb).sort_values('RMSE')
print(pooling_xgb_df.to_string(index=False))

In [ ]:
# === Combined Results ===
print(f"\n{'='*60}")
print("FULL POOLING COMPARISON (MLP vs XGBoost)")
print('='*60)

# Combine MLP and XGBoost results
all_pooling = pooling_results + pooling_results_xgb  # pooling_results = MLP from before
all_pooling_df = pd.DataFrame(all_pooling).sort_values('RMSE')
print(all_pooling_df.to_string(index=False))

---
## Part D — Results & Visualization

In [ ]:
# === D1: Results Table ===
results_df = pd.DataFrame(ALL_RESULTS)
results_df = results_df.sort_values('Test_RMSE')

print("\n" + "="*80)
print("RESULTS: Chronos-2 on C-MAPSS FD001")
print("="*80)
print(results_df.to_string(index=False))

print("\n" + "="*80)
print("BASELINES (from IMSA2025 paper)")
print("="*80)
baselines = [
    {'Model': 'XGBoost (direct)', 'RMSE': 14.64, 'Score': 307.13},
    {'Model': 'Classical LSTM', 'RMSE': 16.14, 'Score': 397.75},
    {'Model': 'Proposed LSTM (IMSA2025)', 'RMSE': 14.32, 'Score': 325.05},
    {'Model': 'Chronos-1 + XGBoost', 'RMSE': 15.93, 'Score': 379.02},
    {'Model': 'Chronos-1 + ANN', 'RMSE': 14.92, 'Score': 641.58},
]
baselines_df = pd.DataFrame(baselines)
print(baselines_df.to_string(index=False))

# Combined comparison
print("\n" + "="*80)
print("FULL COMPARISON (sorted by RMSE)")
print("="*80)

# Best Chronos-2 results per head
best_c2 = results_df[results_df['Feature_Set'] == 'correlation'][['Model', 'Head', 'Test_RMSE', 'Test_Score']]
best_c2 = best_c2.rename(columns={'Test_RMSE': 'RMSE', 'Test_Score': 'Score'})
best_c2['Model'] = 'Chronos-2 + ' + best_c2['Head']
best_c2 = best_c2[['Model', 'RMSE', 'Score']]

# Combine with baselines
full_comparison = pd.concat([best_c2, baselines_df], ignore_index=True)
full_comparison = full_comparison.sort_values('RMSE')
print(full_comparison.to_string(index=False))

# Highlight best
best = full_comparison.iloc[0]
print(f"\n🏆 Best overall: {best['Model']} (RMSE={best['RMSE']}, Score={best['Score']})")

In [ ]:
# === D3: Save Results ===
results_df.to_csv(f'{SAVE_DIR}/chronos2_results_{TIMESTAMP}.csv', index=False)

best = results_df.iloc[0].to_dict()
with open(f'{SAVE_DIR}/best_result_{TIMESTAMP}.json', 'w') as f:
    json.dump(best, f, indent=2, default=str)

# Save full comparison with baselines
full_comparison.to_csv(f'{SAVE_DIR}/full_comparison_{TIMESTAMP}.csv', index=False)

summary = {
    'dataset': 'C-MAPSS FD001',
    'window_size': W,
    'rul_cap': RUL_CAP,
    'chronos2_model': CHRONOS2_MODEL,
    'best_model': best['Model'],
    'best_features': best['Feature_Set'],
    'best_head': best['Head'],
    'best_rmse': best['Test_RMSE'],
    'best_score': best['Test_Score'],
    'heads_tested': ['Ridge', 'MLP', 'XGBoost', 'LSTM', 'BiLSTM'],
    'feature_sets_tested': ['correlation', 'aficv'],
    'baselines': {
        'proposed_lstm_imsa2025': {'rmse': 14.32, 'score': 325.05},
        'xgboost_direct': {'rmse': 14.64, 'score': 307.13},
        'chronos1_xgboost': {'rmse': 15.93, 'score': 379.02}
    }
}
with open(f'{SAVE_DIR}/summary_{TIMESTAMP}.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Results saved to {SAVE_DIR}")
print(f"\n{'='*60}")
print("BEST CHRONOS-2 RESULT:")
print(f"  Features: {best['Feature_Set']}")
print(f"  Head: {best['Head']}")
print(f"  RMSE: {best['Test_RMSE']}")
print(f"  Score: {best['Test_Score']}")
print('='*60)

print(f"\n{'='*60}")
print("COMPARISON WITH BASELINES:")
print(f"  vs Proposed LSTM (14.32):    {'+' if best['Test_RMSE'] > 14.32 else '-'}{abs(best['Test_RMSE'] - 14.32):.2f} RMSE")
print(f"  vs XGBoost direct (14.64):   {'+' if best['Test_RMSE'] > 14.64 else '-'}{abs(best['Test_RMSE'] - 14.64):.2f} RMSE")
print(f"  vs Chronos-1 + XGB (15.93):  {'+' if best['Test_RMSE'] > 15.93 else '-'}{abs(best['Test_RMSE'] - 15.93):.2f} RMSE")
print('='*60)

In [ ]:
# === D5: Best Model (Mean Pooling + MLP) - Prediction vs Reality ===

# Re-extract embeddings with mean pooling for the best combination
fs_name = 'correlation'
data = data_dict[fs_name]

print("Extracting embeddings with MEAN pooling...")
X_train_emb = extract_chronos2_embeddings_v2(chronos2_pipeline, data['X_train'], pooling='mean')
X_test_emb = extract_chronos2_embeddings_v2(chronos2_pipeline, data['X_test'], pooling='mean')
print(f"  Train: {X_train_emb.shape}, Test: {X_test_emb.shape}")

# Train MLP (best head for mean pooling)
print("\nTraining MLP...")
mlp = MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=500,
                   early_stopping=True, random_state=42)
mlp.fit(X_train_emb, data['y_train'])
y_pred = mlp.predict(X_test_emb)

y_test = data['y_test']
units_test = data['units_test']

# Evaluate
rmse, score = evaluate_per_unit(y_test, y_pred, units_test)
print(f"✓ RMSE: {rmse:.2f}, Score: {score:.1f}")

# Get last prediction per unit
def get_last_per_unit(y_true, y_pred, units):
    df = pd.DataFrame({'unit': units, 'true': y_true, 'pred': y_pred})
    last_idx = df.groupby('unit').tail(1).index
    return df.loc[last_idx].sort_values('unit')

results = get_last_per_unit(y_test, y_pred, units_test)
units = results['unit'].values
true_rul = results['true'].values
pred_rul = results['pred'].values

print(f"\nReady to plot: {len(units)} test units")

In [ ]:
# === D6: Plot 1 - Line Graph ===

fig, ax = plt.subplots(figsize=(16, 6))

x = np.arange(len(units))

# Line plots
ax.plot(x, true_rul, 'o-', label='True RUL', color='#2ecc71', linewidth=2, markersize=5, alpha=0.8)
ax.plot(x, pred_rul, 's--', label='Predicted RUL', color='#3498db', linewidth=2, markersize=5, alpha=0.8)

# Fill between to show error
ax.fill_between(x, true_rul, pred_rul, alpha=0.2, color='gray', label='Error')

ax.set_xlabel('Test Unit ID', fontsize=12)
ax.set_ylabel('RUL (cycles)', fontsize=12)
ax.set_title(f'Chronos-2 + MLP (Mean Pooling): True vs Predicted RUL\nRMSE = {rmse:.2f}, Score = {score:.1f}',
             fontsize=14, fontweight='bold')
ax.set_xticks(x[::5])
ax.set_xticklabels(units[::5])
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/best_model_line_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()